In [5]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup
import pandas as pd
import time
import os
from tqdm import tqdm

driver = webdriver.Chrome()

# 검색어
query = "휠체어 장애 -강아지 -고양이 -고소 -청구 -사고" #    ----->  ⭐ 여기 수정 ⭐
  # 예시 : '시각장애 +안경' 검색 -> url에는 'query=시각장애+%2B안경'로 들어감 -> "시각장애+%2B안경"로 query 값 설정
  # 네이버 지식인에 검색해보고 url에서 가져오기 추천
  # https://kin.naver.com/
  # <검색팁> A 에 대한 검색결과 중 B, C를 포함하고 D를 제외하고 싶다면 ->  A +B +C -D  로 검색!

# 기간 구간
periods = [
    ("2026", "2026.01.01.", "2026.05.20."),
    ("2025", "2025.01.01.", "2025.12.31."),
    ("2024", "2024.01.01.", "2024.12.31.")
]

saved_files = []

# =========================
# 연도별 수집 및 저장
# =========================
for year, start_date, end_date in periods:

    print(f"\n==============================")
    print(f"📌 {year}년 데이터 수집 시작")
    print(f"{start_date} ~ {end_date}")
    print(f"==============================")

    period_param = f"{start_date}%7C{end_date}"
    period_label = f"{start_date} ~ {end_date}"

    links = []

    # -------------------------
    # 1. 해당 연도 링크 수집
    # -------------------------
    for page in range(100):

        url = (
            f"https://kin.naver.com/search/list.naver?"
            f"sort=date&query={query}&period={period_param}&section=qna&page={page+1}"
        )

        driver.get(url)
        time.sleep(1.5)

        results = driver.find_elements(By.CSS_SELECTOR, "a._searchListTitleAnchor")

        if len(results) == 0:
            print(f"  → {page+1}페이지 결과 없음, {year}년 링크 수집 종료")
            break

        for r in results:
            link = r.get_attribute("href")

            if link and "detail.naver" in link:
                links.append(link)

        print(f"  {page+1}페이지 완료 / 현재 링크 수: {len(links)}")

    # 링크 중복 제거
    links = list(dict.fromkeys(links))

    print(f"{year}년 중복 제거 후 링크 수: {len(links)}")

    # -------------------------
    # 2. 해당 연도 상세 페이지 크롤링
    # -------------------------
    data = []

    for link in tqdm(links, desc=f"{year}년 지식인 크롤링 진행중"):

        driver.get(link)
        time.sleep(2)

        try:
            # iframe 들어가기
            try:
                driver.switch_to.frame("mainFrame")
            except:
                pass

            soup = BeautifulSoup(driver.page_source, "html.parser")

            # 제목
            try:
                title = soup.select_one(".endTitleSection").get_text(strip=True)
            except:
                title = ""

            # 날짜
            try:
                date = soup.select(".infoItem")[1].get_text(strip=True)
            except:
                date = ""

            # 질문 내용
            try:
                content = soup.select_one(".questionDetail").get_text(" ", strip=True)
            except:
                content = ""

            data.append([year, period_label, title, content, date, link])

        finally:
            driver.switch_to.default_content()

    # -------------------------
    # 3. 연도별 CSV 저장
    # -------------------------
    df_year = pd.DataFrame(
        data,
        columns=["연도", "수집구간", "제목", "내용", "날짜", "링크"]
    )

    filename = f"naver_kin_시각장애_안경_{year}.csv"  # ⭐ 여기 수정 ⭐
    df_year.to_csv(filename, index=False, encoding="utf-8-sig")

    saved_files.append(filename)

    print(f"✅ {year}년 저장 완료:", filename)
    print(f"✅ {year}년 저장 건수:", len(df_year))

# =========================
# 4. 연도별 파일 합치기
# =========================
df_list = []

for file in saved_files:
    temp = pd.read_csv(file, encoding="utf-8-sig")
    df_list.append(temp)

df_all = pd.concat(df_list, ignore_index=True)

# 혹시 모를 전체 중복 제거
df_all = df_all.drop_duplicates(subset=["링크"]).reset_index(drop=True)

final_filename = "naver_kin_시각장애_안경_2020_2026_전체.csv"   # ⭐ 여기 수정 ⭐
df_all.to_csv(final_filename, index=False, encoding="utf-8-sig")

path = os.getcwd()

print("\n==============================")
print("전체 크롤링 완료")
print("연도별 저장 파일:")
for file in saved_files:
    print("-", file)

print("\n전체 통합 파일:", final_filename)
print("파일 저장 위치:", path)
print("전체 데이터 수:", len(df_all))
print("==============================")

driver.quit()


📌 2026년 데이터 수집 시작
2026.01.01. ~ 2026.05.20.
  1페이지 완료 / 현재 링크 수: 10
  2페이지 완료 / 현재 링크 수: 20
  3페이지 완료 / 현재 링크 수: 30
  4페이지 완료 / 현재 링크 수: 40
  5페이지 완료 / 현재 링크 수: 50
  6페이지 완료 / 현재 링크 수: 60
  7페이지 완료 / 현재 링크 수: 70
  8페이지 완료 / 현재 링크 수: 80
  9페이지 완료 / 현재 링크 수: 90
  10페이지 완료 / 현재 링크 수: 100
  11페이지 완료 / 현재 링크 수: 110
  12페이지 완료 / 현재 링크 수: 118
  → 13페이지 결과 없음, 2026년 링크 수집 종료
2026년 중복 제거 후 링크 수: 118


2026년 지식인 크롤링 진행중: 100%|████████████████████████████████████████████████████| 118/118 [05:01<00:00,  2.55s/it]


✅ 2026년 저장 완료: naver_kin_시각장애_안경_2026.csv
✅ 2026년 저장 건수: 118

📌 2025년 데이터 수집 시작
2025.01.01. ~ 2025.12.31.
  1페이지 완료 / 현재 링크 수: 10
  2페이지 완료 / 현재 링크 수: 20
  3페이지 완료 / 현재 링크 수: 30
  4페이지 완료 / 현재 링크 수: 40
  5페이지 완료 / 현재 링크 수: 50
  6페이지 완료 / 현재 링크 수: 60
  7페이지 완료 / 현재 링크 수: 70
  8페이지 완료 / 현재 링크 수: 80
  9페이지 완료 / 현재 링크 수: 90
  10페이지 완료 / 현재 링크 수: 100
  11페이지 완료 / 현재 링크 수: 110
  12페이지 완료 / 현재 링크 수: 120
  13페이지 완료 / 현재 링크 수: 130
  14페이지 완료 / 현재 링크 수: 140
  15페이지 완료 / 현재 링크 수: 150
  16페이지 완료 / 현재 링크 수: 160
  17페이지 완료 / 현재 링크 수: 170
  18페이지 완료 / 현재 링크 수: 180
  19페이지 완료 / 현재 링크 수: 190
  20페이지 완료 / 현재 링크 수: 200
  21페이지 완료 / 현재 링크 수: 210
  22페이지 완료 / 현재 링크 수: 220
  23페이지 완료 / 현재 링크 수: 230
  24페이지 완료 / 현재 링크 수: 240
  25페이지 완료 / 현재 링크 수: 250
  26페이지 완료 / 현재 링크 수: 260
  27페이지 완료 / 현재 링크 수: 270
  28페이지 완료 / 현재 링크 수: 280
  29페이지 완료 / 현재 링크 수: 290
  30페이지 완료 / 현재 링크 수: 300
  31페이지 완료 / 현재 링크 수: 310
  32페이지 완료 / 현재 링크 수: 320
  33페이지 완료 / 현재 링크 수: 330
  34페이지 완료 / 현재 링크 수: 340
  35페이지 완료 / 현재 링크 수: 350
  

2025년 지식인 크롤링 진행중: 100%|████████████████████████████████████████████████████| 478/478 [20:36<00:00,  2.59s/it]


✅ 2025년 저장 완료: naver_kin_시각장애_안경_2025.csv
✅ 2025년 저장 건수: 478

📌 2024년 데이터 수집 시작
2024.01.01. ~ 2024.12.31.
  1페이지 완료 / 현재 링크 수: 10
  2페이지 완료 / 현재 링크 수: 20
  3페이지 완료 / 현재 링크 수: 30
  4페이지 완료 / 현재 링크 수: 40
  5페이지 완료 / 현재 링크 수: 50
  6페이지 완료 / 현재 링크 수: 60
  7페이지 완료 / 현재 링크 수: 70
  8페이지 완료 / 현재 링크 수: 80
  9페이지 완료 / 현재 링크 수: 90
  10페이지 완료 / 현재 링크 수: 100
  11페이지 완료 / 현재 링크 수: 110
  12페이지 완료 / 현재 링크 수: 120
  13페이지 완료 / 현재 링크 수: 130
  14페이지 완료 / 현재 링크 수: 140
  15페이지 완료 / 현재 링크 수: 150
  16페이지 완료 / 현재 링크 수: 160
  17페이지 완료 / 현재 링크 수: 170
  18페이지 완료 / 현재 링크 수: 180
  19페이지 완료 / 현재 링크 수: 190
  20페이지 완료 / 현재 링크 수: 200
  21페이지 완료 / 현재 링크 수: 210
  22페이지 완료 / 현재 링크 수: 220
  23페이지 완료 / 현재 링크 수: 230
  24페이지 완료 / 현재 링크 수: 240
  25페이지 완료 / 현재 링크 수: 250
  26페이지 완료 / 현재 링크 수: 260
  27페이지 완료 / 현재 링크 수: 270
  28페이지 완료 / 현재 링크 수: 280
  29페이지 완료 / 현재 링크 수: 290
  30페이지 완료 / 현재 링크 수: 300
  31페이지 완료 / 현재 링크 수: 310
  32페이지 완료 / 현재 링크 수: 320
  33페이지 완료 / 현재 링크 수: 330
  34페이지 완료 / 현재 링크 수: 340
  35페이지 완료 / 현재 링크 수: 350
  

2024년 지식인 크롤링 진행중: 100%|████████████████████████████████████████████████████| 361/361 [15:40<00:00,  2.61s/it]


✅ 2024년 저장 완료: naver_kin_시각장애_안경_2024.csv
✅ 2024년 저장 건수: 361

전체 크롤링 완료
연도별 저장 파일:
- naver_kin_시각장애_안경_2026.csv
- naver_kin_시각장애_안경_2025.csv
- naver_kin_시각장애_안경_2024.csv

전체 통합 파일: naver_kin_시각장애_안경_2020_2026_전체.csv
파일 저장 위치: /Users/donna/Downloads
전체 데이터 수: 949


In [1]:
%pip install selenium
%pip install webdriver_manager


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: /Users/donna/.local/pipx/venvs/notebook/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: /Users/donna/.local/pipx/venvs/notebook/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
